# LIBERO eval — 이 노드 = **GPU 2장** (3/3)

**이미 학습된 모델**로 LIBERO-10 eval (학습 안 함). 노드 3대 = **GPU 4 / 4 / 2**.
이 노트북은 **2-GPU 노드** 몫만 돌린다. 나머지는 다른 노드에서 → eval_node1_4gpu(4GPU), eval_node2_4gpu(4GPU).

- 전체 = 6모델 × 4 seed = **24 eval**. GPU 비(4:4:2)로 세 노드에 나눔.
- 150k 체크포인트 × **500ep**, action(.pt) 기록 → jerk/LDJ/SPARC/SignFlip.
- 체크포인트 없는 (모델,seed) 나 이미 끝난 eval 은 **자동 skip** → 재실행 안전.
- ⚠️ **LIBERO 시뮬 필요**. 결과 = `eval_clean/libero_10/…` → 리포트가 자동 pooled.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

## 1) 이 노드가 돌릴 eval 목록


In [ ]:
NODE_IDX = 2                 # 0=1번(4GPU) · 1=2번(4GPU) · 2=3번(2GPU)
GPU_COUNTS = [4, 4, 2]          # 세 노드 GPU 수
N_EP = 500

ALL = [(t, s) for s in [0, 1, 2, 3] for t in cf.TRAIN_TAGS]   # 6모델 × 4seed = 24
MINE = cf.split_by_gpu(ALL, GPU_COUNTS)[NODE_IDX]            # 이 노드 몫
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]           # 이 노드 GPU (0번부터)
print(f'이 노드({GPU_COUNTS[NODE_IDX]}GPU) | GPU {gpus} | {len(MINE)} eval × {N_EP}ep')
print('세 노드 분배:', [len(x) for x in cf.split_by_gpu(ALL, GPU_COUNTS)], '(=', sum(GPU_COUNTS), 'GPU 비율)\n')
for t, s in MINE:
    got = cf.resolved_ckpt_step(t, s, step=cf.CKPT_STEP)
    flag = 'OK 150k' if got == cf.CKPT_STEP else ('❌ ckpt 없음' if got is None else f'⚠ 최근접 {got:,}')
    print(f'   {t:10} seed{s}   {flag}')

## 2) 실행 — 500ep, GPU 하나당 eval 하나(OOM 방지)


In [ ]:
cf.run_libero_eval_jobs(MINE, gpus=gpus, n_episodes=N_EP)

## 3) 결과 (SR + 영상 수)


In [ ]:
import glob
print(f"{'MODEL':<12}{'SEED':>5}{'SR':>9}{'VIDEOS':>8}")
print('-' * 34)
for t, s in MINE:
    st = cf.get_eval_status(t, s, 'libero_10')
    sr = f"{st['sr']*100:.1f}%" if st['sr'] is not None else '-'
    vids = glob.glob(str(cf.eval_clean_dir(t, s, 'libero_10') / '**' / '*.mp4'), recursive=True)
    print(f'{t:<12}{s:>5}{sr:>9}{len(vids):>8}')